# Dataset Preparation and Feature Analysis

This notebook prepares the 2022–2025 Formula 1 race data for predicting a driver’s next lap time in seconds, using only information available at the end of the current lap.

I check missing values, create the next-lap target and apply documented filtering rules. The full lap history is saved with an EligiblePair flag for later modelling and evaluation. Missing tyre information is left unchanged until the model inputs are chosen.

The prepared dataset will use 2022–2023 for training, 2024 for validation and 2025 for testing.

## Imports and loading data

In [19]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt 
import seaborn as sns

project_root=Path.cwd()

if project_root.name=="notebooks":
    project_root=project_root.parent

data_dir=project_root / "data"

laps=pd.read_pickle(
    data_dir / "race_laps_2022_2025.pkl"
)

display(
    laps.groupby("Season").agg(
        Races=("RoundNumber", "nunique"),
        LapRecords=("LapNumber", "size")
    )
)

,Races,LapRecords
Season,,
2022,22,23577
2023,22,24420
2024,24,26604
2025,24,26689


## Data Preparation and Quality Checks

In [20]:
laps_prepared=laps.copy()
compound=laps_prepared["Compound"].astype("string").str.strip()

laps_prepared["Compound"]=compound.mask(
    compound.str.lower().isin(["","none","nan","null","<na>"]),
    pd.NA
)

laps_prepared["LapTimeSeconds"]=(
    laps_prepared["LapTime"].dt.total_seconds()
)

laps_prepared["MissingLapTime"] = laps_prepared["LapTime"].isna()
laps_prepared["MissingTyreLife"] = laps_prepared["TyreLife"].isna()
laps_prepared["MissingStint"] = laps_prepared["Stint"].isna()
laps_prepared["MissingPosition"] = laps_prepared["Position"].isna()
laps_prepared["MissingCompound"] = laps_prepared["Compound"].isna()

laps_prepared["IsPitLap"] = (
    laps_prepared["PitInTime"].notna()
    |laps_prepared["PitOutTime"].notna() #or
)

lap_keys=["Season", "RoundNumber","DriverNumber", "LapNumber"]

laps_prepared["DuplicateLapKey"]=laps_prepared.duplicated(
    subset=lap_keys,
    keep=False
)

quality_attributes=[
    "MissingLapTime",
    "MissingTyreLife",
    "MissingStint",
    "MissingPosition",
    "MissingCompound",
    "IsPitLap",
    "DuplicateLapKey"
]

quality_summary=(
    laps_prepared[quality_attributes].sum().rename_axis("Finding").reset_index(name="Count")
)

display(quality_summary)

assert len(laps_prepared) == len(laps)
assert not laps_prepared["DuplicateLapKey"].any()

,Finding,Count
0,MissingLapTime,1539
1,MissingTyreLife,989
2,MissingStint,354
3,MissingPosition,135
4,MissingCompound,780
5,IsPitLap,6741
6,DuplicateLapKey,0


## Next-lap target

In [21]:
group_keys=["Season","RoundNumber","DriverNumber"]
laps_prepared=(
    laps_prepared.sort_values(group_keys + ["LapNumber"]).reset_index(drop=True)
)

driver_races=laps_prepared.groupby(group_keys)

laps_prepared["NextLapNumber"] = (
    driver_races["LapNumber"].shift(-1)
)

laps_prepared["NextLapTimeSeconds"]=(
    driver_races["LapTimeSeconds"].shift(-1)
)

laps_prepared["IsConsecutiveLap"]=(
    laps_prepared["NextLapNumber"] == laps_prepared["LapNumber"] + 1
)

laps_prepared["NextLapTimeSeconds"]=(
    laps_prepared["NextLapTimeSeconds"].where(
        laps_prepared["IsConsecutiveLap"]
    )
)

display(
    laps_prepared[
        [
            "Season",
            "EventName",
            "Driver",
            "LapNumber",
            "LapTimeSeconds",
            "NextLapNumber",
            "NextLapTimeSeconds",
            "IsConsecutiveLap"
        ]
    ].head(15)
)

print(f"Total rows: {len(laps_prepared):,}")
print(
    "Rows with a next-lap target:",
    f"{laps_prepared['NextLapTimeSeconds'].notna().sum():,}"
)

,Season,EventName,Driver,LapNumber,LapTimeSeconds,NextLapNumber,NextLapTimeSeconds,IsConsecutiveLap
0,2022,Bahrain Grand Prix,VER,1.0,100.236,2.0,97.880,True
1,2022,Bahrain Grand Prix,VER,2.0,97.880,3.0,98.357,True
2,2022,Bahrain Grand Prix,VER,3.0,98.357,4.0,98.566,True
3,2022,Bahrain Grand Prix,VER,4.0,98.566,5.0,98.877,True
4,2022,Bahrain Grand Prix,VER,5.0,98.877,6.0,98.940,True
5,2022,Bahrain Grand Prix,VER,6.0,98.940,7.0,99.062,True
6,2022,Bahrain Grand Prix,VER,7.0,99.062,8.0,99.092,True
7,2022,Bahrain Grand Prix,VER,8.0,99.092,9.0,99.251,True
8,2022,Bahrain Grand Prix,VER,9.0,99.251,10.0,99.392,True
9,2022,Bahrain Grand Prix,VER,10.0,99.392,11.0,99.479,True


Total rows: 101,290
Rows with a next-lap target: 97,962


## Selecting lap pairs without Pit Stops
Pit laps do not represent normal lap pace

In [22]:
driver_laps=laps_prepared.groupby(group_keys)
laps_prepared["NextLapIsPitLap"]= (
    driver_laps["IsPitLap"].shift(-1)
)

keep_pair=(
    laps_prepared["IsConsecutiveLap"]
    & laps_prepared["LapTimeSeconds"].notna()
    & laps_prepared["NextLapTimeSeconds"].notna()
    & laps_prepared["IsPitLap"].eq(False)
    & laps_prepared["NextLapIsPitLap"].eq(False)
)

laps_no_pits=laps_prepared.loc[keep_pair].copy()

print(f"Before filtering: {len(laps_prepared):,}")
print(f"After filtering: {len(laps_no_pits):,}")
print(f"Excluded: {len(laps_prepared) - len(laps_no_pits):,}")

Before filtering: 101,290
After filtering: 88,525
Excluded: 12,765


## Selecting lap pairs with track status

In [23]:
driver_laps=laps_prepared.groupby(group_keys)
laps_prepared["NextLapTrackStatus"]=(
    driver_laps["TrackStatus"].shift(-1)
)

keep_status = (
    keep_pair
    & laps_prepared["TrackStatus"].eq("1")
    & laps_prepared["NextLapTrackStatus"].eq("1")
)

laps_status=laps_prepared.loc[keep_status].copy()

print(f"Before filtering: {len(laps_no_pits):,}")
print(f"After filtering: {len(laps_status):,}")
print(f"Excluded: {len(laps_no_pits) - len(laps_status):,}")

Before filtering: 88,525
After filtering: 79,118
Excluded: 9,407


Track Status needs to be 1 always, as it means clear race. Other possible values can be found in "https://docs.fastf1.dev/api_reference/legacy/f1_api.html#fastf1.api.track_status_data". Other values are still important, however they are not important for this specific case.

## Track Conditions and Timing Quality

In [24]:
driver_laps=laps_prepared.groupby(group_keys)
laps_prepared["NextLapIsAccurate"]=(
    driver_laps["IsAccurate"].shift(-1)
)

keep_quality = (
    keep_status
    & laps_prepared["IsAccurate"].eq(True)
    & laps_prepared["NextLapIsAccurate"].eq(True)
)

laps_quality=laps_prepared.loc[keep_quality].copy()

print(f"Before filtering: {len(laps_status):,}")
print(f"After filtering: {len(laps_quality):,}")
print(f"Excluded: {len(laps_status) - len(laps_quality):,}")

Before filtering: 79,118
After filtering: 78,236
Excluded: 882


## Filtering Conclusion

After all of this filtering steps, we got to a data size of 78,236 pairs. The original data had to go through some checks, as it has some missing values, some laps were pit laps, meaning that they were out of the race pace. Then we checked for track status, to find possible yellow flag laps, safety car, red flags or virtual safety car. These two steps cut the data from ~100000 records to ~79000 pairs. I always required consecutive laps also.
I required both laps to pass FastF1’s timing checks by using IsAccurate. This removed another 882 pairs, leaving 78,236 eligible pairs.

## Checking Remaining Missing Values

In [25]:
laps_quality[["Compound", "TyreLife", "Stint"]].isna().sum()

Compound    646
TyreLife    730
Stint       307
dtype: int64

In [26]:
laps_quality[laps_quality["Compound"].isna()].head(20)

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,MissingPosition,MissingCompound,IsPitLap,DuplicateLapKey,NextLapNumber,NextLapTimeSeconds,IsConsecutiveLap,NextLapIsPitLap,NextLapTrackStatus,NextLapIsAccurate
6759,0 days 03:17:29.165000,STR,18,0 days 00:01:24.515000,31.0,5.0,NaT,NaT,0 days 00:00:22.801000,0 days 00:00:39.300000,...,False,True,False,False,32.0,97.937,True,False,1,True
6760,0 days 03:19:07.102000,STR,18,0 days 00:01:37.937000,32.0,5.0,NaT,NaT,0 days 00:00:23.023000,0 days 00:00:51.204000,...,False,True,False,False,33.0,84.605,True,False,1,True
6761,0 days 03:20:31.707000,STR,18,0 days 00:01:24.605000,33.0,5.0,NaT,NaT,0 days 00:00:22.714000,0 days 00:00:39.001000,...,False,True,False,False,34.0,82.704,True,False,1,True
6762,0 days 03:21:54.411000,STR,18,0 days 00:01:22.704000,34.0,5.0,NaT,NaT,0 days 00:00:22.492000,0 days 00:00:38.010000,...,False,True,False,False,35.0,82.363,True,False,1,True
6763,0 days 03:23:16.774000,STR,18,0 days 00:01:22.363000,35.0,5.0,NaT,NaT,0 days 00:00:22.010000,0 days 00:00:38.149000,...,False,True,False,False,36.0,82.264,True,False,1,True
6764,0 days 03:24:39.038000,STR,18,0 days 00:01:22.264000,36.0,5.0,NaT,NaT,0 days 00:00:21.956000,0 days 00:00:38.334000,...,False,True,False,False,37.0,81.741,True,False,1,True
6765,0 days 03:26:00.779000,STR,18,0 days 00:01:21.741000,37.0,5.0,NaT,NaT,0 days 00:00:22.192000,0 days 00:00:37.712000,...,False,True,False,False,38.0,81.665,True,False,1,True
6766,0 days 03:27:22.444000,STR,18,0 days 00:01:21.665000,38.0,5.0,NaT,NaT,0 days 00:00:21.762000,0 days 00:00:38.020000,...,False,True,False,False,39.0,80.557,True,False,1,True
6767,0 days 03:28:43.001000,STR,18,0 days 00:01:20.557000,39.0,5.0,NaT,NaT,0 days 00:00:21.784000,0 days 00:00:37.395000,...,False,True,False,False,40.0,80.222,True,False,1,True
6768,0 days 03:30:03.223000,STR,18,0 days 00:01:20.222000,40.0,5.0,NaT,NaT,0 days 00:00:21.644000,0 days 00:00:37.290000,...,False,True,False,False,41.0,79.988,True,False,1,True


## Decision on missing-values
These missing values will not be filled or removed yet, as they can provide still, some useful information for the model. Handling them will be decided later, on the modelling stage.

In [27]:
laps_quality.groupby("Season").size()

Season
2022    16971
2023    18902
2024    21394
2025    20969
dtype: int64

## Saving Dara

In [28]:
laps_prepared["EligiblePair"] = keep_quality

laps_prepared.to_pickle(
    data_dir / "prepared_race_laps_2022_2025.pkl"
)